In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import wfdb
import random

# Reload the data
def load_beats(record_id):
    try:
        record = wfdb.rdrecord(f'mitdb/{record_id}')
        annotation = wfdb.rdann(f'mitdb/{record_id}', 'atr')
    except:
        return [], []
    signal = record.p_signal[:, 0]
    beat_samples = annotation.sample
    labels = annotation.symbol
    beats, classes = [], []
    for i, sym in enumerate(labels):
        if sym not in ['N', 'V']:
            continue
        r = beat_samples[i]
        if r - 50 < 0 or r + 50 > len(signal):
            continue
        beats.append(signal[r - 50:r + 50])
        classes.append(sym)
    return beats, classes

N_beats, V_beats = [], []
for pid in range(100, 131):
    beats, labels = load_beats(str(pid))
    for b, l in zip(beats, labels):
        if l == 'N': N_beats.append(b)
        elif l == 'V': V_beats.append(b)

# Balance
min_len = min(len(N_beats), len(V_beats))
X = np.array(random.sample(N_beats, min_len) + random.sample(V_beats, min_len))
y = np.array([0]*min_len + [1]*min_len)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
print(f"Data loaded! Train: {len(X_train)}, Test: {len(X_test)}")

Data loaded! Train: 2153, Test: 539


In [3]:
# Reshape for CNN (needs 3D input)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Convert labels to categorical
y_train_cat = to_categorical(y_train, 2)
y_test_cat = to_categorical(y_test, 2)

# Build CNN model
model = Sequential([
    Conv1D(32, kernel_size=5, activation='relu', input_shape=(100, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 96, 32)         │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 48, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 44, 64)         │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 22, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       180,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 191,106 (746.51 KB)

 Trainable params: 191,106 (746.51 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Train the CNN
history = model.fit(
    X_train_cnn, y_train_cat,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8879 - loss: 0.2567 - val_accuracy: 0.9281 - val_loss: 0.2156
Epoch 2/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9506 - loss: 0.1426 - val_accuracy: 0.9513 - val_loss: 0.1513
Epoch 3/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9605 - loss: 0.1169 - val_accuracy: 0.9698 - val_loss: 0.1111
Epoch 4/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9733 - loss: 0.0951 - val_accuracy: 0.9629 - val_loss: 0.0965
Epoch 5/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9721 - loss: 0.0888 - val_accuracy: 0.9675 - val_loss: 0.1054
Epoch 6/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9774 - loss: 0.0774 - val_accuracy: 0.9698 - val_loss: 0.0712
Epoch 7/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9750 - loss: 0.0797 - val_accuracy: 0.9675 - val_loss: 0.0701
Epoch 8/20
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9803 - loss: 0.0639 - val_accuracy: 0.9745 - val_loss